# MobileADAS3D-H1 GT-only 20-epoch health gate

Run top-to-bottom on a Colab GPU. This workflow uses the frozen Vehicle/Pedestrian taxonomy, H1 Hungarian set supervision, durable Google Drive checkpoints/logs, and automatic resume. Teacher distillation is deliberately disabled. The notebook stops after the complete epoch-20 product AP evaluation; do not continue training without review.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime
import json, os, shlex, subprocess, sys
REPO_URL = 'https://github.com/Ali-RT/mobile_adas3d.git'
BRANCH = 'main'
PROJECT_DIR = Path('/content/mobile_adas3d')
DRIVE_DATASET_ROOT = Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT = Path('/content/kitti')
DATASET_VIEW = Path('/content/kitti_h1')
SPLIT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
TAXONOMY_MANIFEST = Path('/content/drive/MyDrive/mobile_adas3d_manifests/kitti_s1_taxonomy_manifest.json')
OUTPUT_DIR = Path('/content/drive/MyDrive/mobile_adas3d_outputs/mobileadas3d_h1_gt_gate20')
R0_SELECTION = Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
BASE_CONFIG = PROJECT_DIR/'configs/kitti_mobileadas3d_h1_gt_gate.yaml'
EDGE_EVIDENCE = PROJECT_DIR/'artifacts/h1_edge_preflight_20260821.json'
RUNTIME_CONFIG_DIR = PROJECT_DIR/'configs/runtime_h1_gt'
RUN_NAME = 'mobileadas3d_h1_gt_gate20'
def run(command, cwd=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    result=subprocess.run(command,cwd=cwd)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_streamed(command,cwd,log_path):
    from collections import deque
    command=[str(x) for x in command]; log_path.parent.mkdir(parents=True,exist_ok=True)
    print('+',shlex.join(command),flush=True); print('Durable log:',log_path,flush=True)
    tail=deque(maxlen=160); env=os.environ.copy(); env['PYTHONUNBUFFERED']='1'
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=p.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))


In [ ]:
# Repository, dependencies, and GPU. Push the required local commits before cloning in Colab.
if not (PROJECT_DIR/'.git').exists(): run(['git','clone','--branch',BRANCH,REPO_URL,PROJECT_DIR])
else:
    run(['git','fetch','origin'],cwd=PROJECT_DIR); run(['git','checkout',BRANCH],cwd=PROJECT_DIR); run(['git','pull','--ff-only','origin',BRANCH],cwd=PROJECT_DIR)
os.chdir(PROJECT_DIR)
run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],cwd=PROJECT_DIR)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Choose Runtime > Change runtime type > GPU')
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=PROJECT_DIR,text=True).strip())
print('GPU:',torch.cuda.get_device_name(0))


In [ ]:
# Use a complete local stage when available; otherwise create a zero-copy canonical Drive view.
def count_files(path,suffix): return sum(1 for p in path.iterdir() if p.is_file() and p.suffix==suffix) if path.is_dir() else 0
def complete(root): return count_files(root/'training/image_2','.png')==7481 and count_files(root/'training/label_2','.txt')==7481 and count_files(root/'training/calib','.txt')==7481
if complete(LOCAL_DATASET_ROOT): DATASET_ROOT=LOCAL_DATASET_ROOT
else:
    aliases={'image_2':['image_2','image_02'],'label_2':['label_2','label_02'],'calib':['calib']}
    (DATASET_VIEW/'training').mkdir(parents=True,exist_ok=True)
    for canonical,candidates in aliases.items():
        source=next((DRIVE_DATASET_ROOT/'training'/name for name in candidates if (DRIVE_DATASET_ROOT/'training'/name).is_dir()),None)
        if source is None: raise FileNotFoundError(f'Missing source for {canonical}: {candidates}')
        link=DATASET_VIEW/'training'/canonical
        if link.is_symlink() and link.resolve()==source.resolve(): continue
        if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace unexpected {link}')
        link.symlink_to(source,target_is_directory=True)
    DATASET_ROOT=DATASET_VIEW
if not complete(DATASET_ROOT): raise RuntimeError(f'KITTI view incomplete: {DATASET_ROOT}')
print('Dataset root:',DATASET_ROOT)
print('Counts:',{name:count_files(DATASET_ROOT/'training'/name,suffix) for name,suffix in [('image_2','.png'),('label_2','.txt'),('calib','.txt')]})


In [ ]:
# Freeze provenance, regenerate the exact taxonomy audit, and execute a real CUDA forward/loss preflight.
for required in (R0_SELECTION,EDGE_EVIDENCE):
    if not required.is_file(): raise FileNotFoundError(required)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
run([sys.executable,'scripts/prepare_h1_gt_gate.py','--base-config',BASE_CONFIG,'--r0-selection',R0_SELECTION,'--edge-evidence',EDGE_EVIDENCE,'--output-dir',OUTPUT_DIR,'--config-dir',RUNTIME_CONFIG_DIR,'--run-name',RUN_NAME],cwd=PROJECT_DIR)
GATE_CONFIG=RUNTIME_CONFIG_DIR/'mobileadas3d_h1_gt_gate20.yaml'
COMMON=['--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,'--output-dir',OUTPUT_DIR]
run([sys.executable,'-m','unittest','tests.test_h1_set_training','tests.test_mobileadas3d_h1','-v'],cwd=PROJECT_DIR)
run_streamed([sys.executable,'-u','scripts/create_kitti_taxonomy_manifest.py','--config',GATE_CONFIG,'--profile','colab_drive','--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR,'--output',TAXONOMY_MANIFEST],PROJECT_DIR,OUTPUT_DIR/'taxonomy_audit.log')
run_streamed([sys.executable,'-u','scripts/check_training_ready.py','--config',GATE_CONFIG,*COMMON,'--require-cuda','--report',OUTPUT_DIR/'training_preflight.json'],PROJECT_DIR,OUTPUT_DIR/'training_preflight.log')
manifest=json.loads((OUTPUT_DIR/'h1_gt_gate_manifest.json').read_text())
assert manifest['architecture']=='MobileADAS3D-H1' and manifest['distillation_enabled'] is False and manifest['criterion']=='h1_hungarian_set'
print(json.dumps(manifest,indent=2))


## Twenty-epoch supervised health gate

This cell automatically resumes only the exact H1 run. It writes every checkpoint and the complete streamed log to Google Drive. Re-running it after a disconnect continues from `latest.pt`.


In [ ]:
candidates=sorted((OUTPUT_DIR/'runs').glob(f'*{RUN_NAME}*/checkpoints/latest.pt'),key=lambda p:p.stat().st_mtime)
RESUME=candidates[-1] if candidates else None
command=[sys.executable,'-u','scripts/train_mobile_adas3d.py','--config',GATE_CONFIG,*COMMON,'--run-name',RUN_NAME]
if RESUME:
    payload=torch.load(RESUME,map_location='cpu',weights_only=False)
    if payload.get('epoch',0)>=20: print(f'Gate already complete at epoch {payload["epoch"]}: {RESUME}')
    else: command += ['--resume',RESUME]
if not RESUME or torch.load(RESUME,map_location='cpu',weights_only=False).get('epoch',0)<20:
    run_streamed(command,PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'train_{RUN_NAME}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
candidates=sorted((OUTPUT_DIR/'runs').glob(f'*{RUN_NAME}*/checkpoints/latest.pt'),key=lambda p:p.stat().st_mtime)
if not candidates: raise FileNotFoundError('No H1 latest checkpoint found')
LATEST_CHECKPOINT=candidates[-1]; TRAIN_RUN_DIR=LATEST_CHECKPOINT.parent.parent
payload=torch.load(LATEST_CHECKPOINT,map_location='cpu',weights_only=False)
if payload.get('epoch')!=20: raise RuntimeError(f'Expected completed epoch 20, got {payload.get("epoch")}')
print('Run directory:',TRAIN_RUN_DIR); print('Checkpoint:',LATEST_CHECKPOINT); print('Best metric:',payload.get('best_metric'))


In [ ]:
# Complete product-taxonomy AP_R40 evaluation. Expect 3,769/3,769 files.
GATE_EVAL_DIR=TRAIN_RUN_DIR/'kitti_r40_gate20'
run_streamed([sys.executable,'-u','scripts/evaluate_kitti_r40.py','--config',GATE_CONFIG,*COMMON,'--checkpoint',LATEST_CHECKPOINT,'--split','val','--score-threshold','0.001','--topk','50','--nms-iou-threshold','0.5','--output-dir',GATE_EVAL_DIR],PROJECT_DIR,OUTPUT_DIR/'colab_logs'/f'eval_{RUN_NAME}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
import pandas as pd
summary=json.loads((GATE_EVAL_DIR/'kitti_r40_summary.json').read_text())
assert summary['complete_split'] and summary['evaluated_images']==3769
metrics=pd.DataFrame(summary['metrics'])
display(metrics.pivot_table(index=['metric','class_name'],columns='difficulty',values='ap_r40').round(3))
print('Send this AP table plus the final epoch-20 training summary before any continuation or distillation.')
